# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

We will explore available record sets and fields, extract tabular data, and perform common exploratory steps.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the FAIR^2 dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print("\nDataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Keywords:", metadata.keywords if hasattr(metadata, 'keywords') else [])

## 2. Data Overview
Let's review available record sets and their fields by their `@id`s.
You may use `dataset.metadata.recordSet` (if present) to discover the record sets. In the FAIR^2 dataset, the record sets are defined in the schema, which you can enumerate and inspect.

In [ ]:
# Discover record sets
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name','[No name]')}, description: {rs.get('description','[No description]')}")
    # List fields in this record set
    fields = rs.get('field', [])
    print("  Fields:")
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"    - Field @id: {f['@id']} | name: {f.get('name','[No name]')} | Type: {f.get('dataType','[No type]')}")
    print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis.
Fields and columns are referenced by their `@id`s.

Below, we extract all record sets and display their columns.

In [ ]:
# Prepare to load each record set
from collections import OrderedDict

dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"\nLoaded RecordSet @id: {rid}")
        print('Columns:', df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for RecordSet @id: {rid}")

## 4. Exploratory Data Analysis (EDA)
Common data processing steps: filtering, normalizing numeric fields, categorizing data.

**Note:** All entity references use their `@id`.

We'll select a numeric field and a group field from the available columns (use the overview above to choose appropriate ones).

In [ ]:
# Example: If RecordSet contains 'age' as a numeric field
for rs_id, df in dataframes.items():
    probable_numeric = [col for col in df.columns if ("age" in col.lower() or "interval" in col.lower() or col.endswith("@id:Age") or col.endswith("@id:Interval"))]
    if probable_numeric:
        numeric_field = probable_numeric[0]
        print(f"Using numeric field {numeric_field} in RecordSet {rs_id}")
        break
else:
    numeric_field = None

if numeric_field:
    # Filtering (Example: age > 50)
    threshold = 50
    rs_id = rs_id  # Use the record set id found above
    filtered_df = dataframes[rs_id][dataframes[rs_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, col_norm]].head())

    # Try to group by a field (e.g., 'sex', or anatomical location)
    probable_group = [col for col in filtered_df.columns if ("sex" in col.lower() or "location" in col.lower() or col.endswith("@id:Sex") or col.endswith("@id:AnatomicalLocation"))]
    group_field = probable_group[0] if probable_group else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found to analyze.")

## 5. Visualization
Explore distributions or relationships between fields. Below, we'll visualize the distribution of the chosen numeric field and its relation to the group field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and rs_id in dataframes:
    df = dataframes[rs_id]
    # Distribution of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field found
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field} in RecordSet {rs_id}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we've explored the FAIR^2 dataset, examining its record sets and fields using their `@id`s, loading records, and performing EDA with filtering and normalization on numeric fields. Visualizations highlighted variable distributions and group relationships where available. For deeper analysis, consult the original Croissant schema for field types and definitions.